# OpenLearn-AI OCR Benchmark — External GPU Execution (PaddleOCR-VL)

ONE benchmark workflow, ONE Engine abstraction. Colab is an execution environment,
not a benchmark layer.

**Run-all flow:** runtime gate → pinned checkout → Python 3.12 `.venv` → Drive assets
(dataset archive + persistent Paddle model cache) → Paddle GPU preflight →
**one-image real inference probe (hard gate)** → existing CLI
`--limit 20 --engine paddle_vl` → automatic validation → automatic archival to Drive.

# PHASE 1 — Runtime

In [ ]:
#@title Configuration { display-mode: "form" }
import os
import subprocess
import threading
import time
from pathlib import Path

# --- pinned inputs ---------------------------------------------------------
ENGINE         = "paddle_vl"      # "docling" remains supported by this same notebook
REPO_URL       = "https://github.com/MuhammadSeyam/OpenLearn-AI.git"
PINNED_COMMIT  = "bebee6d"        # TODO: update to the commit containing this integration before Run All
ARCHIVE_NAME   = "ocrbench-misraj-data-v1.tar.gz"
ARCHIVE_SHA256 = "b66f8e9af44197bf65c2ee0f1c684744e16495390cf87c9fada7fc76af03f7b0"
LIMIT          = 20
# ----------------------------------------------------------------------------

REPO    = Path("/content/OpenLearn-AI")
BENCH   = REPO / "experiments/OCR/ocr-benchmark"
VENV_PY = BENCH / ".venv/bin/python"
DRIVE   = Path("/content/drive/MyDrive/ocrbench")
DIST    = DRIVE / "dist"
MODELS  = DRIVE / "models"
RESULTS_DRIVE = DRIVE / "results"
ARCHIVE = DIST / ARCHIVE_NAME
RESULTS = BENCH / "results/formal/misraj" / ENGINE
MANIFEST = BENCH / "configs/datasets/misraj_DATA_MANIFEST.sha256"

# Source-verified persistent Paddle model cache:
#   paddlex/utils/cache.py:29        CACHE_DIR = env PADDLE_PDX_CACHE_HOME else ~/.paddlex
#   official_models.py:880           save dir = CACHE_DIR / "official_models"
# Set BEFORE any paddle import so the ~1.8 GB of weights lands on Drive.
os.environ["PADDLE_PDX_CACHE_HOME"] = str(MODELS / "paddlex_cache")

def sh(cmd: str, cwd: Path | None = None) -> str:
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}\n{r.stderr[-2000:]}")
    return r.stdout.strip()

def vsh(script: str, name: str) -> str:
    """Run multiline Python with the EXPLICIT .venv interpreter (temp file — never -c)."""
    f = Path(f"/tmp/ocrbench_{name}.py")
    f.write_text(script)
    return sh(f"{VENV_PY} {f}")

class VramWatcher:
    def __init__(self):
        self.peak_mb = 0
        self._stop = threading.Event()
        self._t = threading.Thread(target=self._watch, daemon=True)
    def _watch(self):
        while not self._stop.is_set():
            out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used",
                                  "--format=csv,noheader,nounits"],
                                 capture_output=True, text=True)
            try:
                self.peak_mb = max(self.peak_mb, int(out.stdout.strip()))
            except ValueError:
                pass
            time.sleep(0.5)
    def __enter__(self):
        self._t.start(); return self
    def __exit__(self, *a):
        self._stop.set(); self._t.join(timeout=2)

report = {}
print("configuration ready | engine =", ENGINE)

In [ ]:
print("host python:", __import__("platform").python_version())
import shutil
assert shutil.which("nvidia-smi"), "FAIL: no NVIDIA GPU runtime — Runtime > Change runtime type > GPU"
print(sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader"))
driver = sh("nvidia-smi --query-gpu=driver_version --format=csv,noheader")
assert int(driver.split(".")[0]) >= 550, f"driver {driver} < 550.54 minimum for cu126 wheels"
report["GPU"] = sh("nvidia-smi --query-gpu=name --format=csv,noheader")
report["Driver"] = driver
print("runtime gate PASSED")

# PHASE 2 — Repository

In [ ]:
if REPO.is_dir():
    sh("git fetch origin", cwd=REPO)
else:
    sh(f"git clone {REPO_URL} {REPO}", cwd=Path("/content"))
sh(f"git checkout {PINNED_COMMIT}", cwd=REPO)
head = sh("git rev-parse HEAD", cwd=REPO)
assert head.startswith(PINNED_COMMIT), f"HEAD {head} != pinned {PINNED_COMMIT}"
for rel in ("src/ocrbench/run/run_text.py",
            "src/ocrbench/engines/paddle_vl.py",
            "pyproject.toml", "uv.lock",
            "configs/datasets/misraj_DATA_MANIFEST.sha256"):
    assert (BENCH / rel).exists(), f"missing: {rel}"
report["Repository commit"] = head[:12]
print("checkout verified:", head)

In [ ]:
sh("pip install -q uv")
sh("uv python install 3.12", cwd=BENCH)
sh("uv venv --clear --python 3.12 .venv", cwd=BENCH)
pyver = sh(f"{VENV_PY} --version")
assert pyver.split()[1].startswith("3.12."), pyver
sh("uv sync --frozen", cwd=BENCH)   # core benchmark deps from the repo lockfile only
report["Python"] = pyver.split()[1]
print(".venv:", pyver, "| core deps synced")

# PHASE 3 — Google Drive (persistent storage)

Drive holds three things: the dataset archive (`dist/`), the persistent Paddle
model cache (`models/paddlex_cache/`, honored via the source-verified
`PADDLE_PDX_CACHE_HOME` override), and result archives (`results/`).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
for d in (DIST, MODELS, RESULTS_DRIVE):
    d.mkdir(parents=True, exist_ok=True)
(MODELS / "paddlex_cache").mkdir(parents=True, exist_ok=True)
assert ARCHIVE.is_file(), f"archive missing: {ARCHIVE}"
sidecar = DIST / f"{ARCHIVE_NAME}.sha256"
if sidecar.is_file():
    assert sidecar.read_text().split()[0] == ARCHIVE_SHA256, "Drive sidecar hash mismatch"
print("Drive layout OK | model cache:", os.environ["PADDLE_PDX_CACHE_HOME"])

# PHASE 4 — Dataset (archive SHA-256 → extract → manifest verification)

In [ ]:
import hashlib
digest = hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
if digest != ARCHIVE_SHA256:
    raise SystemExit(f"FAIL: archive hash mismatch\n got {digest}\n want {ARCHIVE_SHA256}")
sh(f"tar -xzf {ARCHIVE} -C {BENCH}")
lines = MANIFEST.read_text().splitlines()
assert lines, "empty manifest"
for line in lines:
    h_exp, s_exp, rel = line.split()
    f = BENCH / rel
    assert f.is_file(), f"missing after extraction: {rel}"
    assert f.stat().st_size == int(s_exp), f"size mismatch: {rel}"
    if hashlib.sha256(f.read_bytes()).hexdigest() != h_exp:
        raise SystemExit(f"FAIL: hash mismatch {rel}")
    print("verified:", rel, f"({s_exp} bytes)")
report["Dataset snapshot"] = ARCHIVE_SHA256[:16] + "..."
print("dataset verification PASSED")

# PHASE 5 — Paddle installation + GPU preflight

Everything stays on the cu12 wheel family so the shared `nvidia/nccl/lib` path
never mixes cu12/cu13 builds; a guard re-checks both framework imports and
repairs once (reinstall torch's NCCL pin) if a collision ever appears.

In [ ]:
# Torch first (the runner uses it for GPU info) — cu126 family to match Paddle.
sh(f"uv pip install --python {VENV_PY} torch "
   f"--index-url https://download.pytorch.org/whl/cu126", cwd=BENCH)
# Paddle GPU wheel (Baidu index) + the OCR/VL doc-parsing stack.
sh(f"uv pip install --python {VENV_PY} paddlepaddle-gpu==3.2.2 "
   f"--index-url https://www.paddlepaddle.org.cn/packages/stable/cu126/ "
   f"--extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match", cwd=BENCH)
sh(f'uv pip install --python {VENV_PY} "paddleocr[doc-parser]>=3.7,<3.8"', cwd=BENCH)

guard = (
    "import torch\n"
    "import paddle\n"
    "print('torch', torch.__version__, '| cuda', torch.cuda.is_available())\n"
    "print('paddle', paddle.__version__, '| compiled_cuda', paddle.device.is_compiled_with_cuda())\n"
)
try:
    print(vsh(guard, "stack_check"))
except RuntimeError as exc:
    if "libtorch_cuda" in str(exc) or "nccl" in str(exc).lower():
        print("NCCL collision detected — repairing torch's NCCL pin, retrying once")
        sh(f"uv pip install --python {VENV_PY} --reinstall nvidia-nccl-cu12", cwd=BENCH)
        print(vsh(guard, "stack_check_retry"))
    else:
        raise
print(sh(f"uv pip list --python {VENV_PY} | grep -iE '^(paddle|torch|nvidia-nccl)'"))

In [ ]:
preflight = (
    "import os\n"
    "import paddle\n"
    "import paddleocr, paddlex\n"
    "assert paddle.device.is_compiled_with_cuda(), 'CPU-only paddle build'\n"
    "assert paddle.device.cuda.device_count() >= 1, 'no CUDA device visible'\n"
    "paddle.utils.run_check()\n"
    "print('paddle:', paddle.__version__)\n"
    "print('paddleocr:', paddleocr.__version__)\n"
    "print('paddlex:', paddlex.__version__)\n"
    "print('model_cache:', os.environ['PADDLE_PDX_CACHE_HOME'])\n"
)
out = vsh(preflight, "paddle_preflight")
print(out)
for line in out.splitlines():
    if line.startswith(("paddle:", "paddleocr:", "paddlex:")):
        key, val = line.split(":", 1)
        report[key.strip().capitalize()] = val.strip()
report["Paddle preflight"] = "PASS (run_check OK, GPU present)"

# PHASE 6 — ONE-IMAGE REAL PROBE (hard gate)

First end-to-end PaddleOCR-VL inference in this project. Observes the REAL
payload structure before any extraction logic is trusted. On failure,
execution stops — the pilot must not start.

In [ ]:
probe_script = """
import json, time
from ocrbench.datasets.misraj import load_misraj
from ocrbench.engines.paddle_vl import PaddleVLEngine

samples = load_misraj()
image = samples[0].image_path          # deterministic first page
engine = PaddleVLEngine(device="gpu:0")

t0 = time.perf_counter(); engine.load(); load_s = time.perf_counter() - t0
t0 = time.perf_counter(); pred = engine.run(image); run_s = time.perf_counter() - t0

assert pred.ok, f"probe inference failed: {pred.error}"
payloads = json.loads(pred.raw_output)
res = payloads[0].get("res", payloads[0])

structure = {
    "top_level_keys": sorted(payloads[0].keys()),
    "res_keys": sorted(res.keys()) if isinstance(res, dict) else None,
    "markdown_type": type(res.get("markdown")).__name__ if isinstance(res, dict) else None,
    "markdown_texts_chars": (
        len(res["markdown"]["texts"])
        if isinstance(res, dict) and isinstance(res.get("markdown"), dict)
        and isinstance(res["markdown"].get("texts"), str) else None),
    "has_rec_texts": isinstance(res, dict) and "rec_texts" in res,
    "has_rec_polys": isinstance(res, dict) and "rec_polys" in res,
}
summary = {
    "sample_id": pred.sample_id, "ok": pred.ok,
    "load_s": round(load_s, 2), "run_s": round(run_s, 2),
    "text_chars": len(pred.text or ""),
    "raw_output_bytes": len(pred.raw_output.encode()),
    "accelerator_device": engine.accelerator_device,
    "model_version": engine.model_version,
    "structure": structure,
}
json.dump(summary, open("/tmp/ocrbench_probe_report.json", "w"), indent=2)
json.dump(payloads[0], open("/tmp/ocrbench_probe_payload.json", "w"), ensure_ascii=False)
print(json.dumps(summary, indent=2))
"""

with VramWatcher() as vw:
    try:
        out = vsh(probe_script, "one_image_probe")
    except RuntimeError as exc:
        print(exc[-2000:])
        raise SystemExit(
            "HARD GATE FAILED: one-image PaddleOCR-VL probe did not succeed — pilot not started"
        )
print(out)
print("peak VRAM MiB during probe:", vw.peak_mb)
report["Probe"] = "PASS"
report["Probe peak VRAM MiB"] = vw.peak_mb

# PHASE 7 — 20-page Pilot (existing benchmark CLI, unchanged)

In [ ]:
existing_runs = ({p.name for p in RESULTS.iterdir()} if RESULTS.is_dir() else set())
pilot_log = sh(
    f"{VENV_PY} -m ocrbench.run.run_text --limit {LIMIT} --engine {ENGINE}",
    cwd=BENCH,
)
print(pilot_log)
new_runs = {p.name for p in RESULTS.iterdir()} - existing_runs
assert len(new_runs) == 1, f"expected exactly one new result directory, got {sorted(new_runs)}"
run_dir = RESULTS / new_runs.pop()
report["Pilot"] = "PASS"
print("new result directory:", run_dir)

# PHASE 8 — Result Validation (automatic)

In [ ]:
import json

raw_files = sorted((run_dir / "raw_outputs").glob("*.json"))
metrics = json.loads((run_dir / "metrics.json").read_text())

for artifact in ("raw_outputs", "metrics.json", "config.yaml", "run_log.md"):
    assert (run_dir / artifact).exists(), f"missing artifact: {artifact}"
assert len(raw_files) == LIMIT, f"expected {LIMIT} raw outputs, got {len(raw_files)}"
assert metrics["dataset"] == "misraj"
assert metrics["engine"] == ENGINE
assert metrics["sample_count"] == LIMIT
assert metrics["successful_samples"] + metrics["failed_samples"] == LIMIT
if metrics.get("accelerator_device") != "cuda":
    raise SystemExit("GPU-FIRST VIOLATION: accelerator_device != cuda")
assert metrics["micro"]["cer_normalized"] >= 0
# every raw output must be parseable JSON
for f in raw_files:
    json.loads(f.read_text())

report["Sample count"] = metrics["sample_count"]
report["Successful samples"] = metrics["successful_samples"]
report["Failed samples"] = metrics["failed_samples"]
report["Accelerator"] = metrics["accelerator_device"].upper()
report["Result validation"] = "PASS"
print(f"validated: run={run_dir.name} raw={len(raw_files)} "
      f"ok={metrics['successful_samples']} failed={metrics['failed_samples']} "
      f"accel={metrics['accelerator_device']} "
      f"micro_cer_norm={metrics['micro']['cer_normalized']:.4f}")

# PHASE 9 — Archive (automatic, to Drive)

In [ ]:
artifact = RESULTS_DRIVE / f"external_result_{run_dir.name}.tar.gz"
assert not artifact.exists(), f"refusing to overwrite existing artifact: {artifact}"
sh(f"tar -czf {artifact} -C {RESULTS} {run_dir.name}")
assert artifact.is_file(), "artifact missing after archiving"
artifact_sha = sh(f"sha256sum {artifact}").split()[0]
report["Result archive"] = str(artifact)
report["SHA256"] = artifact_sha
print("ARTIFACT WRITTEN TO:")
print(artifact)
print("sha256:", artifact_sha)

# PHASE 10 — Final Summary

In [ ]:
line = "=" * 44
order = [
    "Repository commit", "GPU", "Driver", "Python",
    "Paddle", "Paddleocr", "Paddlex",
    "Probe", "Probe peak VRAM MiB", "Pilot",
    "Dataset snapshot",
    "Sample count", "Successful samples", "Failed samples", "Accelerator",
    "Model version",
]
print(line)
print("PADDLEOCR-VL EXTERNAL GPU PILOT")
print(line)
for key in order:
    if key in report:
        print(f"{key}: {report[key]}")
for key in ("Result archive", "SHA256"):
    if key in report:
        print(f"{key}: {report[key]}")
print(line)